# Road PCI Intelligence — Stage 3D: RDD Cross-Domain Evaluation (Models A, B, and C)

This notebook evaluates:
- **Model A**: Masked Baseline (`best.pt`)
- **Model B**: Naive Baseline (`best.pt`)
- **Model C**: Asymmetric Loss Baseline (Seed 42, Epoch 30 `last.pt`)

Evaluation domain: **RDD2022ES Validation Split Only** ($N = 2,560$ images).
Classes from `classes.txt`: 0 longitudinal, 1 longitudinal wide, 2 transverse, 3 transverse wide, 4 alligator, 5 alligator sunken, 6 pothole, 7 pothole deep.


In [ ]:
# ==============================================================================
# 1. ENVIRONMENT & REPOSITORY SETUP
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch==0.5.0', 'albumentations==2.0.8'], check=True)

repo_dir = Path('/kaggle/working/repo')
if not repo_dir.exists():
    print('Cloning repository...')
    subprocess.run(['git', 'clone', 'https://github.com/skilloutofbounds/road-pci-intelligence.git', str(repo_dir)], check=True)
else:
    print('Pulling latest changes...')
    subprocess.run(['git', '-C', str(repo_dir), 'pull'], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import torch
from src.train import seed_everything, log_library_versions, build_model

seed_everything(42)
log_library_versions()
assert torch.cuda.is_available(), 'CRITICAL: CUDA GPU is required!'
device = torch.device('cuda')
print(f'CUDA Device: {torch.cuda.get_device_name(0)}')


In [ ]:
# ==============================================================================
# 2. LOCATE CHECKPOINTS & RESOLVE RDD DATASET
# ==============================================================================
from pathlib import Path
import pandas as pd

k_input = Path('/kaggle/input')

# Model A best.pt
cand_a = list(k_input.rglob('*run_a*/**/best.pt')) + list(k_input.rglob('*masked*/**/best.pt'))
assert len(cand_a) > 0, f'Missing Model A best.pt: {cand_a}'
model_a_pt = cand_a[0]
print(f'Model A Checkpoint: {model_a_pt}')

# Model B best.pt
cand_b = list(k_input.rglob('*run_b*/**/best.pt')) + list(k_input.rglob('*naive*/**/best.pt'))
assert len(cand_b) > 0, f'Missing Model B best.pt: {cand_b}'
model_b_pt = cand_b[0]
print(f'Model B Checkpoint: {model_b_pt}')

# Model C last.pt (Epoch 30 of asymmetric seed 42)
cand_c = list(k_input.rglob('*asymmetric*/**/asymmetric_seed42_last.pt')) + list(k_input.rglob('*asymmetric*/**/last.pt'))
assert len(cand_c) > 0, f'Missing Model C last.pt in /kaggle/input: {cand_c}'
model_c_pt = cand_c[0]
print(f'Model C Checkpoint: {model_c_pt}')

def load_eval_model(weights_p):
    m = build_model(device)
    st = torch.load(weights_p, map_location=device)
    m.load_state_dict(st)
    m.eval()
    return m

model_a = load_eval_model(model_a_pt)
model_b = load_eval_model(model_b_pt)
model_c = load_eval_model(model_c_pt)
print('[OK] Loaded Models A, B, and C successfully.')

# Resolve RDD root
candidates = [
    Path('/kaggle/input/datasets/juusos/rdd2022es'),
    Path('/kaggle/input/rdd2022es'),
    Path('data/rdd2022es')
]
rdd_root = None
for c in candidates:
    if c.exists() and c.is_dir():
        rdd_root = c.resolve()
        break
if rdd_root is None:
    for d in k_input.rglob('rdd2022es'):
        if d.is_dir():
            rdd_root = d.resolve()
            break
assert rdd_root is not None, 'Could not find RDD dataset root!'
print(f'RDD dataset root: {rdd_root}')

# Manifest
manifest_path = repo_dir / 'manifests/rdd2022es_manifest.csv'
assert manifest_path.exists(), f'Missing manifest: {manifest_path}'
df_rdd = pd.read_csv(manifest_path)
df_rdd_val = df_rdd[df_rdd['split'] == 'val'].copy().reset_index(drop=True)
print(f'Filtered to RDD VAL split: {len(df_rdd_val):,} images (Target: 2,560)')
assert len(df_rdd_val) == 2560


In [ ]:
# ==============================================================================
# 3. RDD INFERENCE PIPELINE (MODELS A, B, C)
# ==============================================================================
from PIL import Image
import cv2
import numpy as np
from torch.cuda.amp import autocast
import time

RDD_CLASSES = {
    0: 'longitudinal',
    1: 'longitudinal wide',
    2: 'transverse',
    3: 'transverse wide',
    4: 'alligator',
    5: 'alligator sunken',
    6: 'pothole',
    7: 'pothole deep'
}

def parse_rdd_boxes(txt_path, width, height):
    boxes = []
    if not txt_path.exists():
        return boxes
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(float(parts[0]))
                xc = float(parts[1]) * width
                yc = float(parts[2]) * height
                bw = float(parts[3]) * width
                bh = float(parts[4]) * height
                xmin = max(0, int(round(xc - bw / 2.0)))
                ymin = max(0, int(round(yc - bh / 2.0)))
                xmax = min(width, int(round(xc + bw / 2.0)))
                ymax = min(height, int(round(yc + bh / 2.0)))
                if xmax > xmin and ymax > ymin:
                    boxes.append({
                        'class_id': cid,
                        'box': (ymin, ymax, xmin, xmax),
                        'area': (ymax - ymin) * (xmax - xmin)
                    })
    return boxes

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

@torch.no_grad()
def infer_models_padded(img_rgb):
    h, w = img_rgb.shape[:2]
    pad_h = int(np.ceil(h / 32.0) * 32) - h
    pad_w = int(np.ceil(w / 32.0) * 32) - w
    img_padded = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
    norm = ((img_padded / 255.0) - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(norm).permute(2, 0, 1).unsqueeze(0).float().to(device)
    
    with autocast():
        p_a = torch.sigmoid(model_a(tensor))[0, :, :h, :w].cpu().numpy()
        p_b = torch.sigmoid(model_b(tensor))[0, :, :h, :w].cpu().numpy()
        p_c = torch.sigmoid(model_c(tensor))[0, :, :h, :w].cpu().numpy()
    return p_a, p_b, p_c

print('Running RDD inference across 2,560 validation images for Models A, B, and C...')
t0_eval = time.time()
rdd_records = []

for idx, row in df_rdd_val.iterrows():
    img_p = rdd_root / row['image_path']
    txt_p = rdd_root / row['mask_path_or_label_path'] if row['mask_path_or_label_path'] else Path('')
    
    with Image.open(img_p) as pil:
        img_rgb = np.array(pil.convert('RGB'))
    h, w = img_rgb.shape[:2]
    n_pixels = h * w
    
    boxes = parse_rdd_boxes(txt_p, w, h)
    crack_boxes = [b for b in boxes if 0 <= b['class_id'] <= 5]
    pothole_boxes = [b for b in boxes if 6 <= b['class_id'] <= 7]
    
    pa, pb, pc = infer_models_padded(img_rgb)
    
    pred_a_crk = (pa[0] >= 0.5)
    pred_a_pot = (pa[1] >= 0.5)
    pred_b_crk = (pb[0] >= 0.5)
    pred_b_pot = (pb[1] >= 0.5)
    pred_c_crk = (pc[0] >= 0.5)
    pred_c_pot = (pc[1] >= 0.5)
    
    rdd_records.append({
        'index': idx,
        'image_path': str(img_p),
        'boxes': boxes,
        'crack_boxes': crack_boxes,
        'pothole_boxes': pothole_boxes,
        'n_pixels': n_pixels,
        'a_crack_fg_frac': float(pred_a_crk.sum() / n_pixels),
        'a_pothole_fg_frac': float(pred_a_pot.sum() / n_pixels),
        'a_pred_crack': pred_a_crk,
        'a_pred_pothole': pred_a_pot,
        'b_crack_fg_frac': float(pred_b_crk.sum() / n_pixels),
        'b_pothole_fg_frac': float(pred_b_pot.sum() / n_pixels),
        'b_pred_crack': pred_b_crk,
        'b_pred_pothole': pred_b_pot,
        'c_crack_fg_frac': float(pred_c_crk.sum() / n_pixels),
        'c_pothole_fg_frac': float(pred_c_pot.sum() / n_pixels),
        'c_pred_crack': pred_c_crk,
        'c_pred_pothole': pred_c_pot,
    })
    if (idx + 1) % 500 == 0 or (idx + 1) == len(df_rdd_val):
        print(f'Processed {idx + 1:4d} / {len(df_rdd_val)} images ({time.time() - t0_eval:.1f}s)...')

print(f'Completed RDD cross-domain evaluation in {time.time() - t0_eval:.2f}s.')


In [ ]:
# ==============================================================================
# 4. PRINT COMBINED A / B / C RDD EVALUATION TABLES
# ==============================================================================
# Part A: Zero-Box Images
zero_records = [r for r in rdd_records if len(r['boxes']) == 0]
n_z = len(zero_records)
print('=' * 105)
print(f'A. IMAGES WITH ZERO BOXES (N = {n_z:,} images)')
print('=' * 105)
print(f"{'Model':<24} | {'Channel':<9} | {'Flag > 0.1%':<16} | {'Flag > 0.5%':<16} | {'Flag > 1.0%':<16} | {'Mean FG Frac':<14}")
print('-' * 105)

models = [('a', 'Model A (Masked)'), ('b', 'Model B (Naive)'), ('c', 'Model C (Asymmetric, Ep30)')]

for m_key, m_name in models:
    for ch in ['crack', 'pothole']:
        fracs = np.array([r[f'{m_key}_{ch}_fg_frac'] for r in zero_records])
        r01 = np.mean(fracs > 0.001) * 100
        r05 = np.mean(fracs > 0.005) * 100
        r10 = np.mean(fracs > 0.010) * 100
        m_frac = np.mean(fracs) * 100
        print(f"{m_name:<24} | {ch:<9} | {r01:6.2f}% ({np.sum(fracs > 0.001):4d}) | {r05:6.2f}% ({np.sum(fracs > 0.005):4d}) | {r10:6.2f}% ({np.sum(fracs > 0.010):4d}) | {m_frac:6.4f}%")
    if m_key != 'c':
        print('-' * 105)
print('=' * 105 + '\n')

# Part B: Hit Rate & Box Pixel Coverage
def compute_box_metrics(target_records, box_key, pred_key, cid_filter=None):
    img_hits = []
    box_fracs = []
    for r in target_records:
        boxes = [b for b in r[box_key] if cid_filter is None or b['class_id'] == cid_filter]
        if not boxes:
            continue
        pred_mask = r[pred_key]
        img_has_hit = False
        for b in boxes:
            ymin, ymax, xmin, xmax = b['box']
            area = b['area']
            p_in = pred_mask[ymin:ymax, xmin:xmax].sum()
            frac = float(p_in / area)
            box_fracs.append(frac)
            if frac >= 0.05:
                img_has_hit = True
        img_hits.append(img_has_hit)
    n_i = len(img_hits)
    hr = np.mean(img_hits) * 100 if n_i > 0 else 0.0
    mbf = np.mean(box_fracs) * 100 if box_fracs else 0.0
    return n_i, len(box_fracs), hr, mbf

# Pothole Channel vs Pothole Boxes
pot_records = [r for r in rdd_records if len(r['pothole_boxes']) > 0]
print('=' * 105)
print(f'B. POTHOLE CHANNEL vs POTHOLE BOXES (N = {len(pot_records)} images, 498 boxes)')
print('=' * 105)
print(f"{'Model':<24} | {'Category':<24} | {'Images':<8} | {'Boxes':<8} | {'Hit Rate (>=5%)':<18} | {'Mean Box Frac':<14}")
print('-' * 105)

for m_key, m_name in models:
    n_i, n_b, hr, mbf = compute_box_metrics(pot_records, 'pothole_boxes', f'{m_key}_pred_pothole')
    print(f"{m_name:<24} | {'All Potholes (6-7)':<24} | {n_i:<8} | {n_b:<8} | {hr:6.2f}% ({int(round(hr*n_i/100)):3d}/{n_i}) | {mbf:6.2f}%")
    for cid in [6, 7]:
        n_ic, n_bc, hrc, mbfc = compute_box_metrics(pot_records, 'pothole_boxes', f'{m_key}_pred_pothole', cid_filter=cid)
        c_label = f"  Class {cid} ({RDD_CLASSES[cid]})"
        print(f"{m_name:<24} | {c_label:<24} | {n_ic:<8} | {n_bc:<8} | {hrc:6.2f}% ({int(round(hrc*n_ic/100)):3d}/{n_ic}) | {mbfc:6.2f}%")
    if m_key != 'c':
        print('-' * 105)
print('=' * 105 + '\n')

# Crack Channel vs Crack Boxes
crk_records = [r for r in rdd_records if len(r['crack_boxes']) > 0]
print('=' * 105)
print(f'CRACK CHANNEL vs CRACK BOXES (N = {len(crk_records)} images, 3,278 boxes)')
print('=' * 105)
print(f"{'Model':<24} | {'Category':<24} | {'Images':<8} | {'Boxes':<8} | {'Hit Rate (>=5%)':<18} | {'Mean Box Frac':<14}")
print('-' * 105)

for m_key, m_name in models:
    n_i, n_b, hr, mbf = compute_box_metrics(crk_records, 'crack_boxes', f'{m_key}_pred_crack')
    print(f"{m_name:<24} | {'All Cracks (0-5)':<24} | {n_i:<8} | {n_b:<8} | {hr:6.2f}% ({int(round(hr*n_i/100)):3d}/{n_i}) | {mbf:6.2f}%")
    for cid in range(6):
        n_ic, n_bc, hrc, mbfc = compute_box_metrics(crk_records, 'crack_boxes', f'{m_key}_pred_crack', cid_filter=cid)
        c_label = f"  Class {cid} ({RDD_CLASSES[cid]})"
        print(f"{m_name:<24} | {c_label:<24} | {n_ic:<8} | {n_bc:<8} | {hrc:6.2f}% ({int(round(hrc*n_ic/100)):3d}/{n_ic}) | {mbfc:6.2f}%")
    if m_key != 'c':
        print('-' * 105)
print('=' * 105 + '\n')

# Part C: Cross-Channel Audit
only_crk = [r for r in rdd_records if len(r['crack_boxes']) > 0 and len(r['pothole_boxes']) == 0]
only_pot = [r for r in rdd_records if len(r['pothole_boxes']) > 0 and len(r['crack_boxes']) == 0]
print('=' * 105)
print('C. CROSS-CHANNEL AUDIT')
print(f'  • Images with ONLY Crack boxes:   {len(only_crk):,} images')
print(f'  • Images with ONLY Pothole boxes: {len(only_pot):,} images')
print('=' * 105)
print(f"{'Model':<24} | {'Condition':<25} | {'Evaluated Ch':<14} | {'Flag > 0.1%':<16} | {'Flag > 0.5%':<16} | {'Mean FG Frac':<14}")
print('-' * 105)

for m_key, m_name in models:
    pot_fracs = np.array([r[f'{m_key}_pothole_fg_frac'] for r in only_crk])
    r01_p = np.mean(pot_fracs > 0.001) * 100
    r05_p = np.mean(pot_fracs > 0.005) * 100
    m_p = np.mean(pot_fracs) * 100
    print(f"{m_name:<24} | {'Only Crack Boxes':<25} | {'Pothole':<14} | {r01_p:6.2f}% ({np.sum(pot_fracs > 0.001):4d}) | {r05_p:6.2f}% ({np.sum(pot_fracs > 0.005):4d}) | {m_p:6.4f}%")
    
    crk_fracs = np.array([r[f'{m_key}_crack_fg_frac'] for r in only_pot])
    r01_c = np.mean(crk_fracs > 0.001) * 100
    r05_c = np.mean(crk_fracs > 0.005) * 100
    m_c = np.mean(crk_fracs) * 100
    print(f"{m_name:<24} | {'Only Pothole Boxes':<25} | {'Crack':<14} | {r01_c:6.2f}% ({np.sum(crk_fracs > 0.001):4d}) | {r05_c:6.2f}% ({np.sum(crk_fracs > 0.005):4d}) | {m_c:6.4f}%")
    if m_key != 'c':
        print('-' * 105)
print('=' * 105 + '\n')


In [ ]:
# ==============================================================================
# 5. 12-IMAGE VISUALIZATION (ORIGINAL | MODEL A | MODEL B | MODEL C)
# ==============================================================================
import matplotlib.pyplot as plt
import random

rng = random.Random(42)

clean_cands = [r for r in rdd_records if len(r['boxes']) == 0]
sel_clean = rng.sample(clean_cands, min(4, len(clean_cands)))

pot_cands = [r for r in rdd_records if len(r['pothole_boxes']) > 0]
sel_pot = rng.sample(pot_cands, min(4, len(pot_cands)))

# Classes 0, 1 = longitudinal; Classes 4, 5 = alligator
crk_cands = [r for r in rdd_records if any(b['class_id'] in [0, 1, 4, 5] for b in r['crack_boxes'])]
sel_crk = rng.sample(crk_cands, min(4, len(crk_cands)))

sel_12 = (
    [('Clean', r) for r in sel_clean] +
    [('Pothole', r) for r in sel_pot] +
    [('Crack (Long/Allig)', r) for r in sel_crk]
)

fig, axes = plt.subplots(12, 4, figsize=(20, 36))

for row_idx, (cat_name, r) in enumerate(sel_12):
    with Image.open(r['image_path']) as pil:
        img_rgb = np.array(pil.convert('RGB'))
    
    img_gt = img_rgb.copy()
    for b in r['boxes']:
        ymin, ymax, xmin, xmax = b['box']
        cid = b['class_id']
        color = (0, 100, 255) if cid in [6, 7] else (255, 50, 50)
        cv2.rectangle(img_gt, (xmin, ymin), (xmax, ymax), color, 2)
        c_short = RDD_CLASSES[cid][:6]
        cv2.putText(img_gt, f"{cid}:{c_short}", (xmin, max(15, ymin - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
        
    stem = Path(r['image_path']).name
    if len(stem) > 22:
        stem = stem[:19] + '...'
        
    # Col 0: Image + GT Boxes
    axes[row_idx, 0].imshow(img_gt)
    axes[row_idx, 0].set_title(f"[{cat_name}] {stem} ({len(r['boxes'])} boxes)", fontsize=8.5, fontweight='bold')
    axes[row_idx, 0].axis('off')
    
    # Col 1: Model A
    oa = img_rgb.copy()
    oa[r['a_pred_crack']] = [230, 20, 20]
    oa[r['a_pred_pothole']] = [20, 100, 240]
    axes[row_idx, 1].imshow(oa)
    axes[row_idx, 1].set_title(f"Model A (Masked) | Crk:{r['a_crack_fg_frac']*100:.2f}% Pot:{r['a_pothole_fg_frac']*100:.2f}%", fontsize=8)
    axes[row_idx, 1].axis('off')
    
    # Col 2: Model B
    ob = img_rgb.copy()
    ob[r['b_pred_crack']] = [230, 20, 20]
    ob[r['b_pred_pothole']] = [20, 100, 240]
    axes[row_idx, 2].imshow(ob)
    axes[row_idx, 2].set_title(f"Model B (Naive) | Crk:{r['b_crack_fg_frac']*100:.2f}% Pot:{r['b_pothole_fg_frac']*100:.2f}%", fontsize=8)
    axes[row_idx, 2].axis('off')
    
    # Col 3: Model C
    oc = img_rgb.copy()
    oc[r['c_pred_crack']] = [230, 20, 20]
    oc[r['c_pred_pothole']] = [20, 100, 240]
    axes[row_idx, 3].imshow(oc)
    axes[row_idx, 3].set_title(f"Model C (Asymmetric) | Crk:{r['c_crack_fg_frac']*100:.2f}% Pot:{r['c_pothole_fg_frac']*100:.2f}%", fontsize=8)
    axes[row_idx, 3].axis('off')

plt.subplots_adjust(top=0.98, bottom=0.01, hspace=0.25, wspace=0.08)
out_fig = Path('/kaggle/working/rdd_cross_domain_12_val_abc.png')
plt.savefig(out_fig, dpi=120, bbox_inches='tight')
plt.close()
print(f'[OK] Saved 12-image 4-column figure to {out_fig}')
